# 06 — String Algorithms

## Why This Matters for DSA
Strings are arrays with extra baggage — character-set assumptions, pattern matching, and a genuinely easy-to-miss performance trap in how they're built. This notebook covers the two things that separate "string problems" from "array problems with characters instead of numbers": specialized techniques that exploit a small alphabet (frequency arrays), and substring search algorithms that beat the naive O(n·m) approach — KMP and Rabin-Karp — which reappear constantly in text-processing problems throughout the rest of this course.

## Prerequisites
- `03_STL_Deep_Dive` (Phase 00), Section 4 — `string` basics, `stringstream`
- `01_Arrays_and_Two_Pointers` (Phase 02) — the two-pointer palindrome pattern, extended here
- `01_Complexity_Analysis` (Phase 01) — amortized analysis, reused directly in Section 8

## Learning Objectives
By the end of this notebook, you will be able to:
- Use a fixed-size character frequency array as a faster alternative to a hash map when the alphabet is small and known
- Extend two-pointer palindrome checking to the "at most one removal" variant
- Explain naive pattern matching's O(n·m) worst case with a concrete adversarial example
- Implement KMP's failure function (LPS array) and explain why it lets the text pointer never move backward
- Implement Rabin-Karp's rolling hash and explain why a hash match still requires verification
- Apply expand-around-center to find the longest palindromic substring in O(n²), improving on the O(n³) brute force
- Explain why naive string concatenation can look like an O(n²) trap, and what `reserve()` does and doesn't fix about it
- Choose the right string technique for a new problem instead of defaulting to the first one that comes to mind

## Checklist Before Moving On
- [ ] I can implement a character frequency array solution and explain why it beats a hash map for small alphabets specifically
- [ ] I can trace KMP's LPS array construction by hand on a short pattern
- [ ] I can explain why a Rabin-Karp hash match is not proof of an actual match
- [ ] I can explain both center types (odd and even) that expand-around-center needs to check
- [ ] I can explain the real, measured relationship between `string::operator+=` and `reserve()`
- [ ] I can decide, for a new pattern-matching problem, whether KMP or Rabin-Karp fits better

## Table of Contents
1. Character Frequency Arrays — The O(1)-Lookup Alternative to a Hash Map
2. Palindrome Checks — Two Pointers and the "One Removal" Variant
3. Naive Pattern Matching and Its Real Cost
4. KMP — O(n+m) Pattern Matching via the Failure Function
5. Rabin-Karp — Rolling Hash Pattern Matching
6. Longest Palindromic Substring — Expand Around Center
7. String Building Efficiency
8. Choosing the Right String Technique
9. Phase Checkpoint, Cheat Sheet, and Answer Key
10. LeetCode Practice Problems


## 1. Character Frequency Arrays — The O(1)-Lookup Alternative to a Hash Map

### The Why
`03_Hashing_and_Binary_Search` covered hash maps as the general tool for frequency counting — but for strings limited to a small, known alphabet (lowercase letters, ASCII), a plain fixed-size array indexed directly by character is faster and simpler, with none of a hash map's overhead.

### Core Mechanism
- Declare a fixed-size array (`int freq[26]` for lowercase English letters), and use `c - 'a'` to map each character directly to an array index.
- **Anagram check:** increment the count for every character in the first string, decrement for every character in the second — if the strings are anagrams, every count returns to exactly 0.
- **Why this beats `unordered_map<char,int>`:** array indexing has no hash function to compute, no bucket lookup, no dynamic allocation — the index computation (`c - 'a'`) IS effectively the hash function, and it's exactly as fast as any other array access. This only works because the keyspace (26 lowercase letters, or 128/256 for ASCII) is small and known upfront.

### Common Pitfalls
- Trying to apply this technique to arbitrary Unicode text — a fixed array sized for every possible Unicode code point would be enormous and impractical; a hash map remains the right tool once the alphabet isn't small and bounded.
- Forgetting the length check before comparing frequency counts — two strings of different lengths can never be anagrams, and skipping this check (relying only on the frequency array matching) wastes work in the common case where lengths already disqualify a match.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <string>
using namespace std;

// CHARACTER FREQUENCY ARRAY: for strings limited to a known small alphabet (lowercase
// letters, ASCII, etc.), a plain fixed-size array indexed by character is a faster,
// lower-overhead alternative to a hash map -- O(1) access with no hashing overhead at
// all, and a much smaller constant factor than unordered_map<char,int>.
bool isAnagram(string& s, string& t) {
    if (s.size() != t.size()) return false;   // different lengths can never be anagrams

    int freq[26] = {0};   // index i corresponds to character 'a'+i
    for (char c : s) freq[c - 'a']++;
    for (char c : t) freq[c - 'a']--;

    for (int count : freq) {
        if (count != 0) return false;   // any leftover count means a mismatch
    }
    return true;
}

int main() {
    string s1 = "anagram", t1 = "nagaram";
    string s2 = "rat", t2 = "car";

    cout << "isAnagram(\"anagram\",\"nagaram\") = " << isAnagram(s1, t1) << "\n";
    cout << "isAnagram(\"rat\",\"car\") = " << isAnagram(s2, t2) << "\n";

    // WHY THIS BEATS unordered_map<char,int>: the array has no hash function to compute,
    // no bucket lookup, no dynamic allocation -- indexing IS the hash function, and it's
    // exactly as fast as accessing any other array element. Only works because the
    // "keyspace" (lowercase letters) is small and known in advance -- for arbitrary
    // Unicode strings, a hash map would be the only practical option, since a fixed
    // array sized for every possible Unicode code point would be enormous

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 2. Palindrome Checks — Two Pointers and the "One Removal" Variant

### The Why
Basic two-pointer palindrome checking is already familiar from `01_Arrays_and_Two_Pointers` and `02_Recursion`. This section extends it to a genuinely trickier variant that looks like it might need backtracking but actually resolves with a clean, bounded number of extra checks.

### Core Mechanism
- **Basic check:** converge two pointers from both ends; any mismatch means it's not a palindrome.
- **Valid Palindrome II (at most one removal allowed):** the moment a mismatch is found, there are EXACTLY two possibilities to consider — skip the left character, or skip the right character. Try both remaining ranges as ordinary palindrome checks; if either succeeds, one removal was enough.
- **Why this doesn't need backtracking or trying every possible removal:** the FIRST mismatch pins down exactly which two single-character removals could possibly fix things — trying anything else (removing a character before the first mismatch, or removing both mismatched characters) either wouldn't help or isn't within the "at most one removal" budget.
- **Complexity:** O(n) — the mismatch-driven branch happens at most once, and each of the two follow-up checks is a single O(n) pass, not a recursive or repeated one.

### Common Pitfalls
- Assuming this problem needs to try removing EVERY character one at a time (an O(n²) approach) — the structure of a palindrome mismatch specifically narrows the useful options down to exactly two, and missing that narrowing leads to solving an easy problem the hard way.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <string>
using namespace std;

// basic two-pointer palindrome check -- same pattern as 02_Recursion's recursive version,
// here iterative (the standard real-world form)
bool isPalindrome(const string& s, int left, int right) {
    while (left < right) {
        if (s[left] != s[right]) return false;
        left++;
        right--;
    }
    return true;
}

// VALID PALINDROME II: is s a palindrome, OR can it become one by removing AT MOST
// one character? The moment a mismatch is found, there are exactly two possibilities --
// skip the left character, or skip the right character -- try BOTH and see if either
// resulting range is a clean palindrome.
bool validPalindrome(string s) {
    int left = 0, right = (int)s.size() - 1;
    while (left < right) {
        if (s[left] != s[right]) {
            // one mismatch found -- try skipping either side, exactly once
            return isPalindrome(s, left + 1, right) || isPalindrome(s, left, right - 1);
        }
        left++;
        right--;
    }
    return true;   // no mismatches at all -- already a palindrome, zero removals needed
}

int main() {
    cout << "isPalindrome(\"racecar\") = " << isPalindrome("racecar", 0, 6) << "\n";

    cout << "validPalindrome(\"aba\") = " << validPalindrome("aba") << "\n";       // true, already valid
    cout << "validPalindrome(\"abca\") = " << validPalindrome("abca") << "\n";     // true, remove 'b' or 'c'
    cout << "validPalindrome(\"abc\") = " << validPalindrome("abc") << "\n";       // false, needs 2 removals

    // COMPLEXITY: O(n) -- the mismatch can happen at most once before branching into the
    // two "try skipping" checks, and each of those checks is itself O(n) in the worst
    // case, but they only ever run ONCE (not recursively nested), so total work stays
    // O(n), not O(n^2). This is a good example of a problem that LOOKS like it might need
    // backtracking (try removing different characters) but actually only ever needs to
    // try exactly 2 specific removals, because the FIRST mismatch pins down exactly
    // which two options could possibly work

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. Naive Pattern Matching and Its Real Cost

### The Why
Before KMP and Rabin-Karp make sense as improvements, the naive approach's genuine cost needs to be seen directly — not just labeled "O(n·m) worst case" abstractly, but demonstrated on an input specifically constructed to trigger it.

### Core Mechanism
- Try aligning the pattern at every possible starting position in the text; at each position, compare characters one by one until either a mismatch or a full match.
- **The adversarial case:** a highly repetitive text (`"AAAA...A"`) against a pattern that's ALMOST all matching characters followed by one that never matches (`500` `A`s followed by a `B`). At every single starting position, the comparison gets all the way to the END of the pattern before finally failing — wasting nearly the full pattern length of comparisons, over and over.
- The measured comparison count on this adversarial input matches the `(n-m+1) × m` formula almost exactly, confirming the O(n·m) worst case empirically rather than by assertion alone.

### Common Pitfalls
- Assuming naive pattern matching is "usually fine in practice" without checking whether the expected input could resemble the adversarial shape (repetitive text, near-matching patterns) — this specific shape is common in real text (DNA sequences, repeated log patterns), not just a contrived worst case.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <string>
using namespace std;

// NAIVE PATTERN MATCHING: try aligning the pattern at every possible starting position
// in the text, and check for a full match character by character. Simple, but has a
// genuine worst-case cost worth measuring, not just naming.
long long comparisons = 0;

int naiveSearch(const string& text, const string& pattern) {
    int n = (int)text.size(), m = (int)pattern.size();

    for (int i = 0; i + m <= n; i++) {
        int j = 0;
        while (j < m) {
            comparisons++;
            if (text[i + j] != pattern[j]) break;
            j++;
        }
        if (j == m) return i;   // matched the full pattern starting at i
    }
    return -1;
}

int main() {
    string text1 = "ABABABABABABABABC";
    string pattern1 = "ABABABC";
    comparisons = 0;
    int pos = naiveSearch(text1, pattern1);
    cout << "found at " << pos << ", comparisons=" << comparisons << "\n";

    // WORST CASE: highly repetitive text and pattern -- every alignment gets partway
    // through matching before failing, near the END of the pattern each time, wasting
    // almost the full pattern length of comparisons at EVERY position.
    string text2(1000, 'A');            // "AAAA...A", 1000 A's
    string pattern2(500, 'A');
    pattern2 += 'B';                     // 500 A's followed by a B -- never actually matches

    comparisons = 0;
    int pos2 = naiveSearch(text2, pattern2);
    cout << "adversarial case: found=" << pos2 << ", comparisons=" << comparisons
         << " (text len=" << text2.size() << ", pattern len=" << pattern2.size() << ")\n";

    // COMPLEXITY: O(n*m) worst case -- up to n-m+1 starting positions, each potentially
    // doing up to m comparisons before failing. The measured comparison count above
    // (should be close to (n-m+1)*m for this adversarial case) confirms it directly.
    // KMP (next section) exists SPECIFICALLY to eliminate this redundant re-checking,
    // getting the same answer in O(n+m) instead

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. KMP — O(n+m) Pattern Matching via the Failure Function

### The Why
KMP exists specifically to eliminate naive matching's redundant re-checking — on a mismatch, it never re-examines text characters it's already confirmed, by precomputing exactly how much of a partial match can be reused.

### Core Mechanism
- **The LPS array** (Longest proper Prefix that is also a Suffix): `lps[i]` is the length of the longest prefix of `pattern[0..i]` that is ALSO a suffix of that same substring. This is computed ONCE, from the pattern alone, before any text scanning begins.
- **What the LPS array buys you:** on a mismatch at pattern position `j`, instead of restarting the pattern comparison from position 0 at the next text position (the naive approach), jump directly to `lps[j-1]` — this is the longest prefix of the pattern that's GUARANTEED to still match, based purely on what's already been confirmed, with zero re-comparison needed to prove it.
- **The critical invariant this preserves:** the text pointer `i` NEVER moves backward, ever, throughout the entire search — it advances at most `n` times total, regardless of how many times the pattern pointer `j` falls back via the LPS array. This is precisely what bounds the total work to O(n) for the search itself, plus O(m) to build the LPS array once — O(n+m) total.
- The measured comparison count on the SAME adversarial input from Section 3 (1,500 vs. naive's 250,500) makes the improvement concrete rather than asserted.

### Common Pitfalls
- Building the LPS array incorrectly by not falling back through `lps[len-1]` on a mismatch DURING its own construction — the LPS array's own construction uses the same "fall back, don't restart" logic recursively on itself, and skipping that step produces an incorrect array that silently breaks the whole algorithm's guarantee.
- Confusing "proper prefix/suffix" (which excludes the full string itself) with "any prefix/suffix" (which would trivially include the whole string) — the "proper" qualifier is essential to the definition and the correctness of the technique.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <string>
#include <vector>
using namespace std;

// KMP (Knuth-Morris-Pratt): eliminates the naive algorithm's redundant re-checking by
// precomputing, for the PATTERN alone, how much of a partial match can be REUSED after
// a mismatch -- instead of restarting the pattern from scratch at the next position.

// Build the LPS (Longest Proper Prefix that is also a Suffix) array. lps[i] = the length
// of the longest prefix of pattern[0..i] that is ALSO a suffix of pattern[0..i] --
// this tells us, on a mismatch at position i, how far back in the pattern we can safely
// resume WITHOUT re-checking characters we already know match.
vector<int> buildLPS(const string& pattern) {
    int m = (int)pattern.size();
    vector<int> lps(m, 0);
    int len = 0;   // length of the current longest matching prefix-suffix
    int i = 1;

    while (i < m) {
        if (pattern[i] == pattern[len]) {
            len++;
            lps[i] = len;
            i++;
        } else if (len > 0) {
            len = lps[len - 1];   // fall back to the next-best prefix-suffix length,
                                    // WITHOUT moving i -- reuses previously computed info
        } else {
            lps[i] = 0;
            i++;
        }
    }
    return lps;
}

long long kmpComparisons = 0;

int kmpSearch(const string& text, const string& pattern) {
    int n = (int)text.size(), m = (int)pattern.size();
    if (m == 0) return 0;

    vector<int> lps = buildLPS(pattern);
    int i = 0, j = 0;   // i indexes text, j indexes pattern

    while (i < n) {
        kmpComparisons++;
        if (text[i] == pattern[j]) {
            i++;
            j++;
            if (j == m) return i - j;   // full pattern matched
        } else if (j > 0) {
            j = lps[j - 1];   // KEY DIFFERENCE from naive: jump j back using the LPS
                                // array instead of restarting from j=0 -- `i` NEVER
                                // moves backward, so text is never re-scanned
        } else {
            i++;
        }
    }
    return -1;
}

int main() {
    string text1 = "ABABABABABABABABC";
    string pattern1 = "ABABABC";
    int pos = kmpSearch(text1, pattern1);
    cout << "found at " << pos << ", comparisons=" << kmpComparisons << "\n";

    // same adversarial case as the naive version -- watch the comparison count now
    string text2(1000, 'A');
    string pattern2(500, 'A');
    pattern2 += 'B';

    kmpComparisons = 0;
    int pos2 = kmpSearch(text2, pattern2);
    cout << "adversarial case: found=" << pos2 << ", comparisons=" << kmpComparisons
         << " (text len=" << text2.size() << ", pattern len=" << pattern2.size() << ")\n";

    // COMPLEXITY: O(n + m) -- O(m) to build the LPS array once, O(n) for the search
    // itself, since `i` (the text pointer) NEVER moves backward, ever -- it advances at
    // most n times total across the whole search, regardless of how many times `j` falls
    // back via the LPS array. Compare the comparison count above to the naive version's
    // 250500 on the SAME adversarial input -- KMP should be dramatically lower, close to
    // n+m rather than n*m

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. Rabin-Karp — Rolling Hash Pattern Matching

### The Why
KMP is the go-to for single-pattern search with a clean worst-case guarantee. Rabin-Karp's real strength is different: comparing HASHES instead of full substrings, updated incrementally as the window slides — and its natural extension to searching for MULTIPLE patterns at once, which KMP doesn't handle nearly as gracefully.

### Core Mechanism
- Hash the pattern once. Compute a hash for the FIRST same-length window of the text, then **roll** that hash forward for each subsequent window: remove the outgoing character's contribution, shift the remaining digits up by one position, add the incoming character — all in O(1), without rescanning the window from scratch.
- **A hash match is NOT proof of an actual match.** Different strings can hash to the same value (a collision) — always verify a hash match with a real character-by-character comparison before trusting it. Skipping this verification step is a correctness bug, not just a style choice.
- **Complexity:** O(n+m) AVERAGE case (O(1) rolling update per position, O(m) initial hash computations), but O(n·m) WORST case if hash collisions are frequent, since each collision triggers a full O(m) verification. With a good modulus, collisions are rare enough that average-case behavior dominates in practice.
- **Where Rabin-Karp genuinely wins over KMP:** multi-pattern search. Hash several patterns upfront, then check each rolling window's hash against ALL of them in O(1) per pattern per position — KMP doesn't extend to multiple patterns nearly as directly (that generalization is a different, more involved algorithm, Aho-Corasick, worth knowing exists but outside this notebook's scope).

### Common Pitfalls
- Skipping the verification step after a hash match "because collisions are rare" — rare is not the same as impossible, and this is a genuine correctness bug waiting for an adversarial or just unlucky input, not a performance nitpick.
- Using a modulus that's too small, or a base that shares a common structure with the modulus (like both being powers of the same number) — this can make collisions far MORE frequent than expected, undermining the average-case guarantee.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <string>
using namespace std;

// RABIN-KARP: hash the pattern once, then compute a ROLLING HASH of every same-length
// window in the text -- comparing HASHES is O(1) instead of comparing full substrings.
// The "rolling" part is the key trick: updating the hash for the next window reuses the
// previous window's hash, in O(1), rather than recomputing from scratch.

const long long BASE = 256;         // treat each character as a digit in base-256
const long long MOD = 1000000007;   // large prime modulus, keeps hash values bounded and
                                      // reduces (but doesn't eliminate) collision chance

long long computeHash(const string& s, int len) {
    long long hash = 0;
    for (int i = 0; i < len; i++) {
        hash = (hash * BASE + s[i]) % MOD;
    }
    return hash;
}

int rabinKarpSearch(const string& text, const string& pattern) {
    int n = (int)text.size(), m = (int)pattern.size();
    if (m > n) return -1;

    long long patternHash = computeHash(pattern, m);
    long long windowHash = computeHash(text, m);

    // precompute BASE^(m-1) mod MOD -- needed to "remove" the leading character's
    // contribution when the window slides forward
    long long highestPower = 1;
    for (int i = 0; i < m - 1; i++) highestPower = (highestPower * BASE) % MOD;

    for (int i = 0; ; i++) {
        if (windowHash == patternHash) {
            // HASH MATCH IS NOT PROOF OF A REAL MATCH -- different strings can hash to
            // the same value (a collision). Always verify with an actual character
            // comparison before trusting a hash match.
            if (text.substr(i, m) == pattern) return i;
        }

        if (i + m >= n) break;   // no more full-length windows remain

        // ROLL the hash forward: remove the outgoing character's contribution, shift
        // everything up one digit, add the incoming character -- all O(1), no rescanning
        windowHash = (windowHash - (long long)text[i] * highestPower % MOD + MOD) % MOD;
        windowHash = (windowHash * BASE + text[i + m]) % MOD;
    }
    return -1;
}

int main() {
    string text1 = "ABABABABABABABABC";
    string pattern1 = "ABABABC";
    cout << "found at " << rabinKarpSearch(text1, pattern1) << "\n";

    string text2 = "The quick brown fox jumps over the lazy dog";
    string pattern2 = "lazy";
    cout << "found at " << rabinKarpSearch(text2, pattern2) << "\n";

    string text3 = "aaaa";
    string pattern3 = "bb";
    cout << "found at " << rabinKarpSearch(text3, pattern3) << " (expected -1)\n";

    // COMPLEXITY: O(n+m) AVERAGE case (rolling hash update is O(1) per position, O(n)
    // total positions, plus the O(m) initial hash computations) -- but O(n*m) WORST case
    // if hash collisions are frequent, since each collision triggers a full O(m)
    // character-by-character verification. In practice, with a good modulus, collisions
    // are rare enough that average-case behavior dominates. Rabin-Karp's real strength
    // isn't single-pattern search (KMP is simpler and has a cleaner worst-case guarantee
    // for that) -- it's MULTI-pattern search, since you can hash several patterns and
    // check a rolling window's hash against ALL of them in O(1) per pattern, per position

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. Longest Palindromic Substring — Expand Around Center

### The Why
The brute-force approach to this problem — check every possible substring for being a palindrome — is O(n³). Expand-around-center is the standard improvement, dropping a full factor of n by building each candidate incrementally instead of re-verifying a full substring from scratch every time.

### Core Mechanism
- Every palindrome has a CENTER — a single character for odd-length palindromes, or the gap between two adjacent characters for even-length ones. There are `2n-1` possible centers total (`n` odd-type, `n-1` even-type).
- For each possible center, expand outward as long as the characters on both sides keep matching — this directly builds the palindrome rather than checking a pre-guessed substring.
- **Checking BOTH center types is essential** — a solution that only checks single-character centers will silently miss every even-length palindrome (like `"bb"` or `"abba"`), since those have no single central character at all.
- **Complexity: O(n²)** — `2n-1` centers, each expansion potentially taking O(n) steps in the worst case (a string that's entirely one repeated character expands maximally from every center). This beats the O(n³) brute force by building outward incrementally instead of re-checking a full candidate substring from scratch each time.
- A more advanced O(n) algorithm (Manacher's Algorithm) exists for this exact problem, but is substantially more intricate — worth knowing it exists as the theoretical optimum, not required for fluency at this stage.

### Common Pitfalls
- Forgetting the even-length center case entirely — this is the single most common bug in this pattern, and it silently produces wrong answers on any input whose longest palindrome happens to have even length.
- Off-by-one errors in extracting the final substring bounds — the expansion loop deliberately overshoots by one step in each direction before stopping, so the actual palindrome bounds are one step back inward from where the loop exited.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <string>
using namespace std;

// LONGEST PALINDROMIC SUBSTRING via EXPAND AROUND CENTER: every palindrome has a center
// (a single character for ODD-length palindromes, or a gap between two characters for
// EVEN-length ones) -- try expanding outward from EVERY possible center, tracking the
// longest palindrome found.
pair<int,int> expandFromCenter(const string& s, int left, int right) {
    while (left >= 0 && right < (int)s.size() && s[left] == s[right]) {
        left--;
        right++;
    }
    // loop stops one step too far in both directions -- the actual palindrome bounds
    // are one step back inward from where the loop exited
    return {left + 1, right - 1};
}

string longestPalindrome(string s) {
    if (s.empty()) return "";

    int bestStart = 0, bestLen = 1;

    for (int center = 0; center < (int)s.size(); center++) {
        // ODD-length palindromes: center is a single character
        auto [l1, r1] = expandFromCenter(s, center, center);
        if (r1 - l1 + 1 > bestLen) {
            bestStart = l1;
            bestLen = r1 - l1 + 1;
        }

        // EVEN-length palindromes: center is the gap between center and center+1
        auto [l2, r2] = expandFromCenter(s, center, center + 1);
        if (r2 - l2 + 1 > bestLen) {
            bestStart = l2;
            bestLen = r2 - l2 + 1;
        }
    }

    return s.substr(bestStart, bestLen);
}

int main() {
    cout << longestPalindrome("babad") << "\n";   // "bab" or "aba", both valid (length 3)
    cout << longestPalindrome("cbbd") << "\n";    // "bb"
    cout << longestPalindrome("a") << "\n";       // "a"
    cout << longestPalindrome("racecar") << "\n"; // "racecar"

    // COMPLEXITY: O(n^2) -- there are 2n-1 possible centers (n odd-length, n-1 even-length),
    // and each expansion can take up to O(n) steps in the worst case (a string that's
    // ENTIRELY one repeated character, like "aaaaa", expands maximally from every center).
    // This beats the naive "check every substring for being a palindrome" approach, which
    // is O(n^3) (O(n^2) substrings, O(n) to check each one) -- expand-around-center saves
    // a full factor of n by building each candidate outward incrementally instead of
    // re-verifying a full substring from scratch every time.
    // (A more advanced O(n) algorithm, Manacher's Algorithm, exists for this same problem
    // but is substantially more intricate -- worth knowing it exists, not required here.)

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. String Building Efficiency

### The Why
"Repeated string concatenation is O(n²)" is common advice — worth checking empirically rather than repeating on faith, because the real answer for `std::string` specifically is more nuanced than the blanket statement suggests, and the nuance matters for knowing when `reserve()` is actually solving a real problem versus a theoretical one.

### Core Mechanism
- In principle, `+=` onto a string CAN be O(n) per call if it triggers a reallocation and copy of everything built so far — done `n` times, that's a POTENTIAL O(n²) total, the same underlying concern as `vector`'s reallocation cost from `01_Complexity_Analysis`.
- **In practice, `std::string` (like `std::vector`) uses amortized doubling internally too** — so repeated `+=` in most real implementations is already amortized O(1) per call, NOT a guaranteed O(n) per call. The measured benchmark above shows `reserve()` providing only a modest improvement, not a dramatic one, precisely because the underlying implementation is already doing most of the right thing on its own.
- **What `reserve()` still legitimately buys you:** it avoids ALL reallocations (not just reducing their amortized cost toward zero) when you know the final size upfront — essentially free to call, and there's no real reason not to use it when the final size is known, even if the measured difference is modest with a modern implementation.
- **The genuinely dangerous pattern, worth distinguishing from `+=`:** building a string via `result = result + "x"` (reassignment, not `+=`) in a loop creates a BRAND NEW string on every single iteration by construction — no amortization is possible here regardless of implementation quality, because a fresh allocation and full copy happens every time, unconditionally. This pattern genuinely is O(n²), no caveats.

### Common Pitfalls
- Treating "use `reserve()`" as the fix for a performance problem that's actually caused by `result = result + "x"` reassignment instead of `+=` — `reserve()` helps modestly with `+=`, but the reassignment pattern is a structurally different and much worse problem that reserving capacity doesn't address at all; the real fix there is switching to `+=` (or an equivalent in-place append) in the first place.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <string>
#include <chrono>
using namespace std;
using namespace std::chrono;

// STRING BUILDING EFFICIENCY: repeatedly concatenating onto a std::string with += can
// be O(n) per operation in the worst case (if it triggers a reallocation and copy of the
// existing content, similar in spirit to vector's reallocation from 01_Complexity_Analysis)
// -- done n times, that's a POTENTIAL O(n^2) total, even though each individual += LOOKS
// like a cheap operation.

string buildWithoutReserve(int n) {
    string result;
    for (int i = 0; i < n; i++) {
        result += "x";   // may reallocate + copy the entire existing string so far
    }
    return result;
}

string buildWithReserve(int n) {
    string result;
    result.reserve(n);   // pre-allocate the full capacity upfront -- same idea as
                          // vector::reserve from 03_STL_Deep_Dive
    for (int i = 0; i < n; i++) {
        result += "x";   // never triggers a reallocation -- capacity was already sufficient
    }
    return result;
}

template<typename Func>
double timeIt(Func f, int n) {
    auto start = high_resolution_clock::now();
    volatile size_t len = f(n).size();   // volatile + reading .size() keeps the compiler
    (void)len;                            // from optimizing the whole call away
    auto end = high_resolution_clock::now();
    return duration<double, milli>(end - start).count();
}

int main() {
    cout << "n\twithout reserve (ms)\twith reserve (ms)\n";
    for (int n : {100000, 200000, 400000}) {
        double t1 = timeIt(buildWithoutReserve, n);
        double t2 = timeIt(buildWithReserve, n);
        cout << n << "\t" << t1 << "\t\t\t" << t2 << "\n";
    }

    // IN PRACTICE: std::string, like std::vector, actually uses AMORTIZED doubling
    // internally too -- so repeated += is usually amortized O(1) per operation in real
    // implementations, same reasoning as 01_Complexity_Analysis's vector push_back
    // argument, NOT a guaranteed O(n) per call. reserve() still helps by avoiding ALL
    // reallocations (not just reducing their amortized cost to near-zero), and is
    // virtually free when you know the final size upfront -- there's no reason not to
    // use it in that situation, even though the difference may be small in practice with
    // a modern std::string implementation. The REAL danger case is naive string
    // concatenation via `result = result + "x"` (not `+=`) in a loop for some other
    // language/library where this creates a BRAND NEW string every single iteration --
    // that pattern genuinely is O(n^2) with no amortization possible, since a new
    // allocation happens on every single operation by construction, not just occasionally

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 8. Choosing the Right String Technique

### The Why
This notebook covered several genuinely different tools — closing the loop on which one fits a new problem is the actual transferable skill.

### Core Mechanism
- **Reach for a frequency array when:** the alphabet is small and known (lowercase letters, ASCII), and you need character counting, anagram checks, or similar — faster and simpler than a hash map in this specific situation.
- **Reach for two-pointer palindrome checking when:** the problem is fundamentally about symmetry around a center or matching from both ends — extend to the "one removal" branching technique (Section 2) when a problem allows a small, bounded amount of flexibility.
- **Reach for KMP when:** you need single-pattern search with a guaranteed O(n+m) worst case, no exceptions — the right default choice for pattern matching when you're not sure which technique to reach for.
- **Reach for Rabin-Karp when:** you need to search for MULTIPLE patterns simultaneously, or the problem is naturally framed in terms of comparing substrings by content (like finding duplicate substrings) where a rolling hash directly helps — remember the mandatory verification step for any hash match.
- **Reach for expand-around-center when:** the problem is specifically about palindromic substrings and you need something better than brute force but don't need Manacher's Algorithm's O(n) optimum.
- **Always consider `reserve()` when building a string of a KNOWN final size in a loop** — free, simple, and removes any doubt about reallocation cost, even if the practical difference is often modest.

### Common Pitfalls
- Defaulting to Rabin-Karp for single-pattern search out of familiarity — KMP has a strictly better worst-case guarantee for that specific case and no collision-verification overhead; Rabin-Karp's advantages are specifically about multi-pattern search or hash-based substring comparison, not single-pattern search in general.


## 9. Phase Checkpoint, Cheat Sheet, and Answer Key

### String Technique Cheat Sheet

| Technique | Complexity | Requires | Example |
|---|---|---|---|
| Frequency array | O(n) | Small, known alphabet | Valid Anagram |
| Two-pointer palindrome | O(n) | — | Valid Palindrome |
| One-removal palindrome | O(n) | At most one allowed removal | Valid Palindrome II |
| Naive pattern match | O(n·m) worst case | — | Baseline for comparison |
| KMP | O(n+m) guaranteed | Single pattern | Implement strStr() |
| Rabin-Karp | O(n+m) average, O(n·m) worst | Single or multi-pattern, hash verification | Repeated DNA Sequences |
| Expand around center | O(n²) | — | Longest Palindromic Substring |

### Checkpoint Questions
1. Why does a character frequency array outperform a hash map specifically, and when does that advantage disappear?
2. In Valid Palindrome II, why is checking only two specific removal options sufficient, rather than trying every possible single-character removal?
3. What specific input structure makes naive pattern matching hit its O(n·m) worst case, concretely?
4. What does KMP's LPS array actually represent, and how does it let the text pointer avoid ever moving backward?
5. Why is a Rabin-Karp hash match not sufficient proof of a real match, and what must always follow it?
6. Why must expand-around-center check BOTH odd and even-length centers, and what specifically breaks if only one type is checked?

### Answer Key
1. A frequency array avoids all hash-function computation and bucket lookup — indexing directly via the character (`c - 'a'`) IS the "hash," at the cost of a normal array access rather than a hash table operation. This advantage disappears once the alphabet is large or unbounded (arbitrary Unicode), where a fixed array would need to be impractically large, and a hash map becomes the only practical option.
2. The first mismatch encountered specifically identifies exactly two characters (left or right at that mismatch point) whose removal COULD possibly resolve it — removing any other character either doesn't address this specific mismatch or exceeds the "at most one removal" budget. No other removal candidate needs to be considered.
3. A highly repetitive text combined with a pattern that matches almost entirely but fails only at its very end (e.g. many repeated characters followed by one that breaks the pattern) — every alignment gets nearly all the way through the pattern before failing, repeating nearly the full pattern length of wasted comparisons at every single position.
4. `lps[i]` is the length of the longest PROPER prefix of `pattern[0..i]` that's also a suffix of that same substring. On a mismatch, falling back to `lps[j-1]` tells you exactly how much of the pattern is GUARANTEED to still match without needing to re-verify it against the text — because that guarantee comes purely from the pattern's own internal structure, computed once in advance. This is what lets the text pointer stay put (never move backward) while only the pattern pointer adjusts.
5. Different strings can hash to the same value (a collision) — a matching hash is necessary but not sufficient evidence of an actual match. A direct character-by-character comparison must always follow a hash match before treating it as a confirmed result.
6. Odd-length palindromes have a single character at their center (like the middle 's' in "aba"); even-length palindromes have no single central character at all — their center is the GAP between two characters (like the gap between the two 'b's in "abba"). Checking only odd centers silently misses every even-length palindrome in the input, producing an incorrect (too-short, or entirely wrong) answer whenever the true longest palindrome happens to have even length.

### Mini Project
Implement `findAnagrams(s, p)`: find the starting indices of all anagrams of `p` within `s`.
1. Build a frequency array (Section 1) for the pattern `p`.
2. Use a FIXED-SIZE sliding window (`02_Sliding_Window_and_Prefix_Sum`, Section 1) of length `len(p)` over `s`, maintaining a running frequency array for the current window as it slides.
3. At each window position, compare the window's frequency array to the pattern's — O(alphabet size) per comparison, or track a running "how many character counts currently match" number to make each check O(1) instead.

This exercises: combining Section 1's frequency-array technique with the fixed-size sliding window pattern from Phase 02's earlier notebook — a direct example of how string algorithms frequently combine with patterns from outside this notebook specifically, rather than existing in isolation.


## 10. LeetCode Practice Problems

Grouped by pattern. Hints point at the pattern's key decision, not the full solution.

### Frequency Arrays and Basic String Manipulation

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 242 | Valid Anagram | Easy | Section 1, directly |
| 49 | Group Anagrams | Medium | Already in `03_Hashing_and_Binary_Search` — also solvable using a frequency-array-derived key instead of a sorted string |
| 387 | First Unique Character in a String | Easy | Frequency array first pass to count, second pass to find the first index with count 1 |

### Palindrome Techniques

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 125 | Valid Palindrome | Easy | Already in `01_Arrays_and_Two_Pointers` — listed here as the direct prerequisite to #680 |
| 680 | Valid Palindrome II | Easy | Section 2, directly |
| 5 | Longest Palindromic Substring | Medium | Section 6, directly |
| 647 | Palindromic Substrings | Medium | Same expand-around-center mechanism as #5, but COUNT every valid expansion instead of tracking only the longest |

### Pattern Matching

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 28 | Find the Index of the First Occurrence in a String | Easy | Implement KMP (Section 4) or Rabin-Karp (Section 5) directly — this problem exists specifically to make you implement substring search from scratch |
| 187 | Repeated DNA Sequences | Medium | Rabin-Karp's rolling hash, or a frequency-map of fixed-length substrings — a natural fit for hash-based substring comparison over many overlapping windows |
| 459 | Repeated Substring Pattern | Easy | A clever KMP-adjacent trick: check if `s` appears within `(s+s)` after removing its own first and last character — worth researching WHY this works as an exercise in creative reuse of pattern-matching machinery |

### Self-Check Before Moving to `07_Linked_Lists`
If you can look at a new pattern-matching problem and decide between KMP (single pattern, guaranteed worst case) and Rabin-Karp (multi-pattern, or hash-based substring comparison) without hesitating, and you remember to check BOTH center types on any palindrome-substring problem, you've internalized this notebook's two most failure-prone details. `07_Linked_Lists` moves away from contiguous-memory structures entirely — a good moment to notice how much of what you've built so far (two pointers, fast/slow pointers from `01_Arrays_and_Two_Pointers`, even KMP-style incremental pointer movement) is about to reappear in a pointer-based, non-contiguous setting.
